# Cost optimisation algorithms

**Data sources:** 

* production - solarEdge (ID:1567917)
* consumption - mvm (GroupID: 4)
* price - HUPX.hu



Using cassandra to store the aforementioned sources after transformations: 
* **consumer_profile**
*  **solar_panel_production**
*  **hupx_energy_price**

Data is stored in 15 minute time intervals. production and consumption is stored in **kwh**, price is in **mwh**



In [37]:
spark.version

'3.5.0'

In [1]:
import findspark
import itertools
import pyspark.sql
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, abs, weekofyear, to_timestamp, col, to_utc_timestamp, count, max, monotonically_increasing_id, lit, concat, expr, regexp_replace,lpad
 

findspark.init()

spark = SparkSession.builder \
    .appName("CassandraConncection") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0")\
    .config("spark.sql.catalog.client","com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.client.spark.cassandra.connection.host","127.0.0.1")\
    .getOrCreate()
    #.config("spark.cassandra.connection.port", "9042")\
    


24/06/28 10:24:00 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.129 instead (on interface ens33)
24/06/28 10:24:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/student/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c1aaef1-16c9-4e47-81da-bc24ee9ba27a;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3

# Extract


In [ ]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show()
price.show()






# Transform

Transformations are needed: All timezones should be casted to **UTC**, granuality should be set to 15 minutes to all sources

production profile:

In [ ]:
#yearly 50 kw
spark.conf.set("spark.sql.session.timeZone", "UTC") # necessary config to aviod clock shift
production_final = production.withColumn("timestamp",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                        .withColumn("profile_id", lit(1567917))\
                        .withColumn("production_(W)", col("production_(W)")*50/10.8*0.00025)\
                        .withColumnRenamed("production_(W)", "production_kwh")\
                        .select("profile_id","timestamp","production_kwh")
                        
num = production_final.toPandas()["production_kwh"].sum()/50/2
print(num)


consumption profile:

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
cons_final = consumption.withColumn("time", to_timestamp(concat(col("Dátum"), lit(" "), col("Negyedórák")), "yyyy.MM.dd HH:mm"))\
                        .withColumn("utc_time", to_utc_timestamp(col("time"), "Europe/Budapest"))\
                        .withColumn("consumption_kwh", col("Group#4").cast("double")*10)\
                        .withColumn("rownum", monotonically_increasing_id())

cons_filter = cons_final.groupBy("utc_time").agg(count("*").alias("count")).filter(col("count")>=2)
cons_filtered = cons_final.join(cons_filter,"utc_time","inner").groupBy("utc_time").agg(max("rownum").alias("rownum"))\
                          .withColumn("modified_utc_time", expr("utc_time + INTERVAL 1 HOUR"))\
                          .select("modified_utc_time","rownum")
cons_final = cons_final.join(cons_filtered,"rownum","left_outer")\
                       .withColumn("timestamp", expr("IFNULL(modified_utc_time, utc_time)"))\
                       .withColumn("profile_id", lit(4))\
                       .select("profile_id","timestamp","consumption_kwh")

cons_final.show()

price:

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
price_final = price.withColumn("Hours",regexp_replace("Hours","[HB]",""))\
                   .withColumn("Hours", expr("cast(Hours as int) - 1"))\
                   .withColumn("Quarters",expr("explode(array(':00',':15',':30',':45'))"))\
                   .withColumn("Hours", concat(lpad("Hours",2,"0"),"Quarters"))\
                   .withColumn("time",to_timestamp(concat("Delivery day",lit(" "),"Hours"),"dd.MM.yyyy HH:mm"))\
                   .withColumn("utc_timestamp", to_utc_timestamp("time", "Europe/Budapest"))\
                   .withColumn("rownum", monotonically_increasing_id())

price_filter = price_final.groupBy("utc_timestamp").agg(count("*").alias("count")).filter(col("count")>=2)
price_filtered = price_final.join(price_filter,"utc_timestamp","inner").groupBy("utc_timestamp").agg(max("rownum").alias("rownum"))\
                            .withColumn("modified_utc_timestamp", expr("utc_timestamp + INTERVAL 1 HOUR")).select("modified_utc_timestamp","rownum")
price_final = price_final.join(price_filtered,"rownum","left_outer")\
                         .withColumn("timestamp", expr("IFNULL(modified_utc_timestamp, utc_timestamp)"))\
                         .withColumnRenamed("Prices (EUR/Mwh)","price_eur_per_mwh")\
                         .select("timestamp","price_eur_per_mwh")

                   

price_final.show()
price_final.printSchema()
price_final.count()

# Load

In [ ]:
cons_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="consumption_profile", keyspace="energycom") \
    .save()

In [ ]:
production_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="production_profile", keyspace="energycom") \
    .save()

In [ ]:
price_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .save()

# Using csv data (optional)

In [ ]:
consumption = cons_final
production = production_final
price = price_final

# Reading from Cassandra DB


In [2]:
consumption = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_profile", keyspace="energycom") \
    .load()
production = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="production_profile", keyspace="energycom") \
    .load()
price = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .load()

consumption.show()
production.show()
price.show()

+-------------------+----------+-------------------+
|          timestamp|profile_id|    consumption_kwh|
+-------------------+----------+-------------------+
|2022-10-12 06:00:00|         4|0.34847000000000006|
|2022-05-03 19:30:00|         4|0.16691999999999999|
|2023-03-12 20:15:00|         4|0.16726999999999997|
|2022-06-22 01:15:00|         4|            0.12324|
|2022-01-08 21:00:00|         4|            0.18702|
|2022-10-02 19:15:00|         4|             0.1547|
|2023-11-05 23:30:00|         4|0.14432999999999999|
|2022-12-27 13:15:00|         4|            0.61922|
|2023-10-07 02:45:00|         4|            0.13576|
|2023-06-17 22:15:00|         4|            0.13513|
|2023-10-25 10:30:00|         4|            0.77376|
|2023-12-21 13:00:00|         4|            0.66542|
|2022-07-18 11:00:00|         4|0.28060999999999997|
|2022-09-22 08:00:00|         4|            0.68465|
|2022-09-03 22:00:00|         4|            0.12425|
|2023-11-10 12:15:00|         4|            0.

# Combining data

In [3]:
from pyspark.sql.functions import *

# filter
consumption   = consumption.filter(col("profile_id") == 4)
production = production.filter(col("profile_id") == 1567917)

# join
process = consumption.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#sort and cast to Pandas
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2022-01-01 01:00:00,0.18187,0.0,41.330002
1,2022-01-01 01:15:00,0.18158,0.0,41.330002
2,2022-01-01 01:30:00,0.18087,0.0,41.330002
3,2022-01-01 01:45:00,0.18103,0.0,41.330002
4,2022-01-01 02:00:00,0.18172,0.0,43.220001


# Optimistaion strategies

Goal: minimise **energy costs**

Scenarios:
* pv only
* greedy strategy
* rule based strategy full
* rule based strategy without LT

# PV only

There is no BESS in this scenario, if there is a sufficit, we sell it to the grid, if there is a power deficit, we buy from the grid.

In [ ]:
pv = process.assign(feeding_grid = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                    taking_from_grid   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                    sell_price = lambda x: x["feeding_grid"]*x["price_eur_per_mwh"]*0.00025,
                    buy_price = lambda x: x["taking_from_grid"]*x["price_eur_per_mwh"]*0.00025,
                    net_revenue = lambda x: x["sell_price"] - x["buy_price"])

pv.head()

In [ ]:
pv_total_generation = pv.assign(generation = lambda x: x['production_kwh'])['generation'].sum()
pv_total_export = pv["feeding_grid"].sum()
pv_self_consumption_rate = 1 - pv_total_export/pv_total_generation

pv_total_consumption = pv.assign(consumption = lambda x: x['consumption_kwh'])['consumption'].sum()
pv_total_import = pv['taking_from_grid'].sum()
pv_self_sufficiency_rate = 1 - pv_total_import/pv_total_consumption

total_cost = pv['buy_price'].sum()
total_revenue = pv['sell_price'].sum()
net_revenue = pv['net_revenue'].sum()

print(f"Total generation: {pv_total_generation}")
print(f"Total consumption: {pv_total_consumption}")
print(f"Self consumption rate: {pv_self_consumption_rate}")
print(f"Self sufficiency_rate: {pv_self_sufficiency_rate}")
print(f"Total cost: {total_cost}")
print(f"Total revenue: {total_revenue}")
print(f"net revenue: {net_revenue}")


# Greedy strategy

If production covers consumption, then surplus is stored into BESS, if it is full, then the remainder is sold to the grid.
If production does not cover consumption, then the deficit is first covered from BESS, and if the energy stored in BESS was not sufficient, then power is bought from the grid.



In [ ]:
Capacity = 100
Charge_rate = 50
Discharge_rate = 50
Capacity_min = 0


In [ ]:
import numpy as np
# calculating energy surplus and deficit from the difference of production and consumption,
# SOC colums are added for the beggining and end of 15 minute intervals, setting default value to 0. 
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# (optional) setting the starting state of the battery:
# basic_p.at[0,"battery_charge_start"] = 10

#calculating the amount we can store of the surplus generated
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        basic_p.at[0,'battery_charge_end'] = __builtins__.min(temp,basic_p.at[0,'battery_charge_start'] + Charge_rate, Capacity)
       
            
#calculating how much of the deficit we can cover from our battery
else:
        temp = basic_p.at[0,'battery_charge_start']  - __builtins__.min(basic_p.at[0,'energy_to_get'], Discharge_rate)
        if temp > Capacity_min:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = Capacity_min

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= Capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = Capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - __builtins__.min(basic_p.at[i,'energy_to_get'], Discharge_rate)
        if temp > Capacity_min:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = Capacity_min
#basic_p.tail(10)

# from the data previously calculated, we determine how much energy we used from our own storage and from the grid,
# furthermore how much energy we feed our system or the grid.
# last we calculate the energy price and the revenue. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.00025) 

basic_p.head(100)



In [ ]:
simple_total_generation = basic_p.assign(generation = lambda x: x['production_kwh'])['generation'].sum()
simple_total_export = basic_p["feeding_grid"].sum()
simple_self_consumption_rate = 1 - simple_total_export/simple_total_generation

simple_total_consumption = basic_p.assign(consumption = lambda x: x['consumption_kwh'])['consumption'].sum()
simple_total_import = basic_p['taking_from_grid'].sum()
simple_self_sufficiency_rate = 1 - simple_total_import/simple_total_consumption

simple_total_cost = basic_p[basic_p['price']>0]['price'].sum()
simple_total_revenue = basic_p[basic_p['price']<0]['price'].sum()*(-1)
simple_net_revenue = basic_p['price'].sum()*(-1)

print("Total generation: ",simple_total_generation, " kw")
print(f"Total consumption: {simple_total_consumption}")
print(f"Self consumption rate: {simple_self_consumption_rate}")
print(f"Self sufficiency rate: {simple_self_sufficiency_rate}")
print(f"Total cost: {simple_total_cost}")
print(f"Total revenue: {simple_total_revenue}")
print(f"Net revenue: {simple_net_revenue}")

# Rule Based Strategy

If there is a deficit, we check whether the energy price reaches the current upper price threshold. If not we buy from the grid, if yes then we use the backup stored in BESS, if we can't cover demand from BESS, then power is bought from the grid.

For surplus, we load as much energy as we can into BESS, the remainder is sold to the grid.

In both cases we check whether the energy price is below the lower price threshold, then we charge the battery as much as we can from the grid.

The upper threshold is calculated from the upper (.25,.5,.75) percentile of the daily forcasted price data.
The lower threshold is calculated from the lower .25 percentile of the previous 15 days' price data.

both thresholds are recalculated daily.

In [ ]:
#function for determining season
def get_season(timestamp):
    # Alakítsuk át a timestamp-et datetime objektummá
    month = timestamp.month
    day = timestamp.day
    
    if (month == 12 and day >= 21) or (1 <= month <= 2) or (month == 3 and day <= 20):
        return 'Winter'
    elif (month == 3 and day >= 21) or (4 <= month <= 5) or (month == 6 and day <= 20):
        return 'Spring'
    elif (month == 6 and day >= 21) or (7 <= month <= 8) or (month == 9 and day <= 21):
        return 'Summer'
    else:
        return 'Autumn'

season_udf = udf(get_season, StringType())

In [ ]:
percentiles = {'Winter':0.2,'Spring':0.1,'Summer':0.1,'Autumn':0.1}

In [ ]:
percentiles = {'Winter':0.5,'Spring':0.3,'Summer':0.1,'Autumn':0.3}

rule_based = process.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                            Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
n_ut = 1
n_lt = 1

#first row
UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')
LT = 0

if rule_based.at[0,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[0,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < Capacity):
               rule_based.at[0,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[0,"SOC_start"],Charge_rate])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[0,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"Deficit"], rule_based.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"] - rule_based.at[0,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
                
    # Surplus side            
else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[0,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[0,"Surplus"] < Charge_rate:
                
                rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"Surplus"], Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
                if rule_based.at[0,"price_eur_per_mwh"] < LT:
                    rule_based.at[0,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[0,"P_P2B"], Capacity - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]

#P_P2L

#P_P2G
#P_G2B
#P_G2L


rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
rule_based.at[0,"Buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
rule_based.at[0,"Selling_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025

for i in range(1,int(rule_based.size/16)):
    # setting SOC 
    rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based.size/16):
        UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
    if (n_lt % 96) & (n_lt >= 1440):
        LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.05, interpolation='lower')
    n_ut += 1
    n_lt += 1

    # comparing demand and production
    # Deficit side
    if rule_based.at[i,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[i,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < Capacity):
               rule_based.at[i,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[i,"SOC_start"],Charge_rate])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[i,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"] - rule_based.at[i,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
                
    # Surplus side            
    else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[i,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[i,"Surplus"] < Charge_rate:
                
                rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"Surplus"], Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
                if rule_based.at[i,"price_eur_per_mwh"] < LT:
                    rule_based.at[i,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[i,"P_P2B"], Capacity - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"]

    #executing the calculations
    rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
    rule_based.at[i,"Buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
    rule_based.at[i,"Selling_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0
LT=0


In [ ]:
rule_total_generation = rule_based["production_kwh"].sum()
rule_total_export = rule_based["P_P2G"].sum()
rule_self_consumption_rate = 1 - rule_total_export/rule_total_generation

rule_total_consumption = rule_based["consumption_kwh"].sum()
rule_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
rule_self_sufficiency_rate = 1 - rule_total_import/rule_total_consumption

rule_total_cost = rule_based["Buy_price"].sum()
rule_total_revenue = rule_based["Selling_price"].sum()
rule_net_revenue = rule_total_revenue - rule_total_cost

print("Total generation: ",rule_total_generation, " kw")
print(f"Total consumption: {rule_total_consumption}")
print(f"Self consumption rate: {rule_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_self_sufficiency_rate}")
print(f"Total cost: {rule_total_cost}")
print(f"Total revenue: {rule_total_revenue}")
print(f"Net revenue: {rule_net_revenue}")

# Rule Based Strategy with no LT

same strategy with one key difference: no lower threshold is used to buy cheaper energy.

In [ ]:
rule_based_no_lt = process.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, x['production_kwh'] - x['consumption_kwh'], 0),                       Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
n_ut = 1


#first row
UT = rule_based_no_lt.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')


if rule_based_no_lt.at[0,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[0,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"]

        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based_no_lt.at[0,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based_no_lt.at[0,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[0,"Deficit"], rule_based_no_lt.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"] - rule_based_no_lt.at[0,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"Deficit"]
                
    # Surplus side            
else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"consumption_kwh"]

        # is BESS full?
        if rule_based_no_lt.at[0,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based_no_lt.at[0,"Surplus"] < Charge_rate:
                
                rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[0,"Surplus"], Capacity - rule_based_no_lt.at[0,"SOC_start"])
                rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"Surplus"] - rule_based_no_lt.at[0,"P_P2B"]
                
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based_no_lt.at[0,"SOC_start"])
                rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"Surplus"] - rule_based_no_lt.at[0,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]

#P_P2L

#P_P2G
#P_G2B
#P_G2L


rule_based_no_lt.at[0,"SOC_end"] = rule_based_no_lt.at[0,"SOC_start"] + rule_based_no_lt.at[0,"P_P2B"] + rule_based_no_lt.at[0,"P_G2B"] - rule_based_no_lt.at[0,"P_B2L"]
rule_based_no_lt.at[0,"Buy_price"] = (rule_based_no_lt.at[0,"P_G2B"] + rule_based_no_lt.at[0,"P_G2L"])*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025
rule_based_no_lt.at[0,"Selling_price"] = rule_based_no_lt.at[0,"P_P2G"]*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025

for i in range(1,int(rule_based_no_lt.size/16)):
    # setting SOC 
    rule_based_no_lt.at[i,"SOC_start"] = rule_based_no_lt.at[i-1,"SOC_end"]
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based_no_lt.size/16):
        UT = rule_based_no_lt.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
    
    n_ut += 1


    # comparing demand and production
    # Deficit side
    if rule_based_no_lt.at[i,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[i,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"]

            #cheaper than LT and there is space
            
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based_no_lt.at[i,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based_no_lt.at[i,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[i,"Deficit"], rule_based_no_lt.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"] - rule_based_no_lt.at[i,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"Deficit"]
                
    # Surplus side            
    else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"consumption_kwh"]

        # is BESS full?
        if rule_based_no_lt.at[i,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based_no_lt.at[i,"Surplus"] < Charge_rate:
                
                rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[i,"Surplus"], Capacity - rule_based_no_lt.at[i,"SOC_start"])
                rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"] - rule_based_no_lt.at[i,"P_P2B"]
                
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based_no_lt.at[i,"SOC_start"])
                rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"] - rule_based_no_lt.at[i,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"Surplus"]

    #executing the calculations
    rule_based_no_lt.at[i,"SOC_end"]       = rule_based_no_lt.at[i,"SOC_start"] + rule_based_no_lt.at[i,"P_P2B"] + rule_based_no_lt.at[i,"P_G2B"] - rule_based_no_lt.at[i,"P_B2L"]
    rule_based_no_lt.at[i,"Buy_price"]     = (rule_based_no_lt.at[i,"P_G2B"] + rule_based_no_lt.at[i,"P_G2L"])*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
    rule_based_no_lt.at[i,"Selling_price"] = rule_based_no_lt.at[i,"P_P2G"]*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0



In [ ]:
rule_no_lt_total_generation = rule_based_no_lt["production_kwh"].sum()
rule_no_lt_total_export = rule_based_no_lt["P_P2G"].sum()
rule_no_lt_self_consumption_rate = 1 - rule_no_lt_total_export/rule_no_lt_total_generation

rule_no_lt_total_consumption = rule_based_no_lt["consumption_kwh"].sum()
rule_no_lt_total_import = rule_based_no_lt["P_G2L"].sum() + rule_based_no_lt["P_G2B"].sum()
rule_no_lt_self_sufficiency_rate = 1 - rule_no_lt_total_import/rule_no_lt_total_consumption

rule_no_lt_total_cost = rule_based_no_lt["Buy_price"].sum()
rule_no_lt_total_revenue = rule_based_no_lt["Selling_price"].sum()
rule_no_lt_net_revenue = rule_no_lt_total_revenue - rule_no_lt_total_cost

print("Total generation: ",rule_no_lt_total_generation, " kw")
print(f"Total consumption: {rule_no_lt_total_consumption}")
print(f"Self consumption rate: {rule_no_lt_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_no_lt_self_sufficiency_rate}")
print(f"Total cost: {rule_no_lt_total_cost}")
print(f"Total revenue: {rule_no_lt_total_revenue}")
print(f"Net revenue: {rule_no_lt_net_revenue}")

# Experiment for rule based hyper parametering

getting seasonal data:

In [ ]:
seasonal_data = production.withColumn("season",season_udf(col("timestamp"))).withColumn("week", weekofyear(col("timestamp"))).withColumn("year",year(col("timestamp")))
avg_data= seasonal_data.groupBy("season").agg(avg("production_kwh").alias("avg_production"))
seasonal_data = seasonal_data.join(avg_data,"season","inner")\
                             .groupBy("season","week").agg(avg("production_kwh").alias("weekly_avg"),max("avg_production").alias("seasonal_avg"),max("year").alias("year"))\
                             .withColumn("diff",abs(col("seasonal_avg")-col("weekly_avg")))
window_spec = Window.partitionBy("season").orderBy(col("diff"))

closest_weeks = seasonal_data.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

closest_weeks.select("season","year","week","seasonal_avg","weekly_avg","diff").show()


extracting the chosen weeks according to the results:

In [ ]:
test_data = spark.createDataFrame(process).withColumn("season",season_udf(col("timestamp")))\
                      .withColumn("week", weekofyear(col("timestamp")))\
                      .withColumn("year",year(col("timestamp")))\
                   .filter((col("year") == 2023) & ((col("week") == 5) | (col("week") == 17) | (col("week") == 36) | (col("week") == 45)))
test_data = test_data.toPandas()
test_data.head()



testing the different percentile variations:

In [ ]:
values = [0.1,0.2,0.3,0.4,0.5]
variations = list(itertools.product(values,repeat=4))


In [ ]:
dataset = []
c = 0
columns = ["percentiles", "self_consumption_rate","self_sufficiency_rate","total_cost","total_revenue","net_revenue"]
for v in variations:
    percentiles['Winter'] = v[0]
    percentiles['Spring'] = v[1]
    percentiles['Summer'] = v[2]
    percentiles['Autumn'] = v[3]
    rule_based = test_data.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                            Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')
    LT = 0
    
    if rule_based.at[0,"Deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[0,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < Capacity):
                   rule_based.at[0,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[0,"SOC_start"],Charge_rate])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[0,"SOC_start"] > Capacity_min:
    
                    #if rule_based.at[i,"Deficit"] < Discharge_rate:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                    rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"Deficit"], rule_based.at[0,"SOC_start"]- Capacity_min, Discharge_rate])
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"] - rule_based.at[0,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
                    
        # Surplus side            
    else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[0,"SOC_start"] < Capacity:
    
                # Is surplus < Charge rate
                if rule_based.at[0,"Surplus"] < Charge_rate:
                    
                    rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"Surplus"], Capacity - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                    
                    if rule_based.at[0,"price_eur_per_mwh"] < LT:
                        rule_based.at[0,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[0,"P_P2B"], Capacity - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]
    
    #P_P2L
    
    #P_P2G
    #P_G2B
    #P_G2L
    
    
    rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
    rule_based.at[0,"Buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    rule_based.at[0,"Selling_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    
    for i in range(1,int(rule_based.size/16)):
        # setting SOC 
        rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based.size/16):
            UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.05, interpolation='lower')
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # Deficit side
        if rule_based.at[i,"Deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[i,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < Capacity):
                   rule_based.at[i,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[i,"SOC_start"],Charge_rate])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[i,"SOC_start"] > Capacity_min:
    
                    #if rule_based.at[i,"Deficit"] < Discharge_rate:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min, Discharge_rate])
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"] - rule_based.at[i,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
                    
        # Surplus side            
        else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[i,"SOC_start"] < Capacity:
    
                # Is surplus < Charge rate
                if rule_based.at[i,"Surplus"] < Charge_rate:
                    
                    rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"Surplus"], Capacity - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                    
                    if rule_based.at[i,"price_eur_per_mwh"] < LT:
                        rule_based.at[i,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[i,"P_P2B"], Capacity - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"]
    
        #executing the calculations
        rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
        rule_based.at[i,"Buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
        rule_based.at[i,"Selling_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0
    r_total_generation = rule_based["production_kwh"].sum()
    r_total_export = rule_based["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based["consumption_kwh"].sum()
    r_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based["Buy_price"].sum()
    r_total_revenue = rule_based["Selling_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    string_percentiles = ','.join([str(value) for value in percentiles.values()])
    dataset.append((string_percentiles, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

    c = c +1
    print(c)
    #print(percentiles.values())
    #print(f"Self consumption rate: {r_self_consumption_rate}")
    #print(f"Self sufficiency rate: {r_self_sufficiency_rate}")
    #print(f"Total cost: {r_total_cost}")
    #print(f"Total revenue: {r_total_revenue}")
    #print(f"Net revenue: {r_net_revenue}")

df = spark.createDataFrame(dataset, columns)
df.show()

In [ ]:
df = df.repartition(1)
df.write.format("csv").option("header","true").option("delimiter",";").mode("overwrite").save("../output/percentiles_177k_10000_kwh_per_y.csv")

In [ ]:
df.orderBy("total_cost").show()

# Linear Programming 

In [12]:
import pyomo.environ as pyo
import pandas
import numpy as np
from pyomo.opt import (SolverFactory, TerminationCondition)

SOC_MIN = 0
SOC_MAX = 50
chargeRate = 12.5
dischargeRate = 12.5
start = 0

In [17]:
dates = process
dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
dates["date"] = dates["timestamp"].dt.date
days =  dates['date'].unique().tolist()

data = process
data["timestamp"] = pandas.to_datetime(data["timestamp"])
data = data[data["timestamp"].dt.date == days[365]]
data.reset_index(drop=True, inplace=True)
t = data.shape[0]

def cost_model(data, t, start):
    con= {}
    gen= {}
    price = {}
    for i in range(0,t):
        con[i+1] = data["consumption_kwh"][i]
        gen[i+1] = data["production_kwh"][i]
        price[i+1] = data["price_eur_per_mwh"][i]*0.00025
    
    model = pyo.ConcreteModel()

    model.T = pyo.RangeSet(t)
    model.M = __builtins__.max(chargeRate,dischargeRate) +1

    model.cin = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.cout = pyo.Var(model.T, within= pyo.NonNegativeReals)
    model.ch = pyo.Var(model.T, within= pyo.NonNegativeReals)
    model.dch = pyo.Var(model.T, within= pyo.NonNegativeReals) 
    model.soc = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(SOC_MIN, SOC_MAX))
    model.soc0 = pyo.Param(within= pyo.NonNegativeReals, default=0, initialize=start)

    model.y_ch = pyo.Var(model.T, within=pyo.Binary)
    model.y_dch = pyo.Var(model.T, within=pyo.Binary)

    
    model.con = pyo.Param(model.T, initialize=con, mutable=True)
    model.gen = pyo.Param(model.T, initialize=gen, mutable=True)
    model.price = pyo.Param(model.T, initialize=price, mutable=True)

    def obj_rule(m):
        #return __builtins__.sum(m.cin[t]*m.price[t] - m.cout[t]*m.price[t] for t in m.T)
        return __builtins__.sum(m.cin[t]*m.price[t] for t in m.T)
    model.obj = pyo.Objective(rule=obj_rule)

    def battery_link_rule(m,t):
        if t == m.T.first():
            return m.soc[t] == m.soc0 -m.dch[t] + m.ch[t]
        return m.soc[t] == m.ch[t] - m.dch[t] + m.soc[t-1]
    model.battery_link_con = pyo.Constraint(model.T, rule= battery_link_rule)
    
    def charge_rate_rule(m,t):
        return m.ch[t] <= chargeRate*m.y_ch[t]
    model.charge_rate_con = pyo.Constraint(model.T, rule= charge_rate_rule)
    def discharge_rate_rule(m,t):
        return m.dch[t] <= dischargeRate*m.y_dch[t]
    model.discharge_rate_con = pyo.Constraint(model.T, rule= discharge_rate_rule)
    def ch_dch_rule(m, t):
        return m.y_ch[t] + m.y_dch[t] <= 1
    model.ch_dch_con = pyo.Constraint(model.T, rule= ch_dch_rule)


    def balance_rule(m,t):
        return m.ch[t] + m.con[t] + m.cout[t] == m.dch[t] + m.gen[t] + m.cin[t]
    model.balance_con = pyo.Constraint(model.T, rule= balance_rule)

    return model

model = cost_model(data, t, start)

solver = pyo.SolverFactory('glpk')
result = solver.solve(model)
#pyo.assert_optimal_termination(result)
#print(pyo.value(model.obj))

#model.con.pprint()
#model.gen.pprint()
#model.price.pprint()
#model.ch.pprint()
#model.dch.pprint()
#model.soc.pprint()
#model.cin.pprint()
#model.cout.pprint()

# Backup algorithm


In [14]:
dates = process
dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
dates["date"] = dates["timestamp"].dt.date
days =  dates['date'].unique().tolist()

data = process
data["timestamp"] = pandas.to_datetime(data["timestamp"])
data = data[data["timestamp"].dt.date == days[365]]
data.reset_index(drop=True, inplace=True)
def backup_greedy(data, start):

# calculating energy surplus and deficit from the difference of production and consumption,
# SOC colums are added for the beggining and end of 15 minute intervals, setting default value to 0. 
    data = data.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))


    data.at[0,"battery_charge_start"] = start

#calculating the amount we can store of the surplus generated
    if data.at[0,'energy_to_store'] > 0:
        temp = data.at[0,'battery_charge_start'] + data.at[0,'energy_to_store']
        data.at[0,'battery_charge_end'] = __builtins__.min(temp, data.at[0,'battery_charge_start'] + chargeRate, SOC_MAX)
       
            
#calculating how much of the deficit we can cover from our battery
    else:
        temp = data.at[0,'battery_charge_start']  - __builtins__.min(data.at[0,'energy_to_get'], dischargeRate)
        if temp > SOC_MIN:
            data.at[0,'battery_charge_end'] = temp
        else:
            data.at[0,'battery_charge_end'] = SOC_MIN

    for i in range(1,data.shape[0]):
        data.at[i,'battery_charge_start'] = data.at[i-1,'battery_charge_end']
        if data.at[i,'energy_to_store'] > 0:
            temp = data.at[i,'battery_charge_start'] + data.at[i,'energy_to_store']
            if temp <= SOC_MAX:
                data.at[i,'battery_charge_end'] = temp
            else:
                data.at[i,'battery_charge_end'] = SOC_MAX
        else:
            temp = data.at[i,'battery_charge_start'] - __builtins__.min(data.at[i,'energy_to_get'], dischargeRate)
            if temp > SOC_MIN:
                data.at[i,'battery_charge_end'] = temp
            else:
                data.at[i,'battery_charge_end'] = SOC_MIN
    
    # from the data previously calculated, we determine how much energy we used from our own storage and from the grid,
    # furthermore how much energy we feed our system or the grid.
    # last we calculate the energy price and the revenue. 
    data = data.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                           feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                           taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                           feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                           price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.00025) 
    output = {
                "cin":data["taking_from_grid"].tolist(),
                "cout":data["feeding_grid"].tolist(),
                "soc":data["battery_charge_end"].tolist(),
                "ch":data["feeding_battery"].tolist(),
                "dch":data["taking_from_battery"].tolist()
             }
    return output

output = backup_greedy(data, start)

96

In [18]:
SOC_MIN = 0
SOC_MAX = 50
chargeRate = 12.5
dischargeRate = 12.5
start = 0

error_count=0
# collecting all dates into a list of unique days.
dates = process
dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
dates["date"] = dates["timestamp"].dt.date
days =  dates['date'].unique().tolist()


linear_programming = process
linear_programming["timestamp"] = pandas.to_datetime(linear_programming["timestamp"])

SOC = []
ch = []
dch = []
c_in = []
c_out = []
#iterate over each day from the database
for day in days:
    
    
    data = linear_programming[linear_programming["timestamp"].dt.date == day]
    data.reset_index(drop=True, inplace=True)
    t = data.shape[0]
    
    model = cost_model(data, t, start)
    solver = pyo.SolverFactory('glpk')
    result = solver.solve(model)
    if pyo.check_optimal_termination(result):
        SOC.extend([model.soc[t].value for t in model.T])
        ch.extend([model.ch[t].value for t in model.T])
        dch.extend([model.dch[t].value for t in model.T])
        c_in.extend([model.cin[t].value for t in model.T])
        c_out.extend([model.cout[t].value for t in model.T])
    else:
        output = backup_greedy(data, start)
        
        SOC.extend(output["soc"])
        ch.extend(output["ch"])
        dch.extend(output["dch"])
        c_in.extend(output["cin"])
        c_out.extend(output["cout"])
        
        error_count += 1
    start = SOC[-1]
print("Job's done!")
print(f"errors: {error_count}")

Job's done!
errors: 19


In [19]:
linear_programming["soc"] = SOC
linear_programming["ch"] = ch
linear_programming["dch"] = dch
linear_programming["c_in"] = c_in
linear_programming["c_out"] = c_out
linear_programming["Buy_price"] = linear_programming["c_in"]*linear_programming["price_eur_per_mwh"]*0.00025
linear_programming["Selling_price"] = linear_programming["c_out"]*linear_programming["price_eur_per_mwh"]*0.00025

In [20]:
lp_total_generation = linear_programming["production_kwh"].sum()
lp_total_export = linear_programming["c_out"].sum()
lp_self_consumption_rate = 1 - lp_total_export/lp_total_generation

lp_total_consumption = linear_programming["consumption_kwh"].sum()
lp_total_import = linear_programming["c_in"].sum()
lp_self_sufficiency_rate = 1 - lp_total_import/lp_total_consumption

lp_total_cost = linear_programming["Buy_price"].sum()
lp_total_revenue = linear_programming["Selling_price"].sum()
lp_net_revenue = lp_total_revenue - lp_total_cost

print("Total generation: ",lp_total_generation, " kw")
print(f"Total consumption: {lp_total_consumption}")
print(f"Self consumption rate: {lp_self_consumption_rate}")
print(f"Self sufficiency rate: {lp_self_sufficiency_rate}")
print(f"Total cost: {lp_total_cost}")
print(f"Total revenue: {lp_total_revenue}")
print(f"Net revenue: {lp_net_revenue}")


Total generation:  131531.75  kw
Total consumption: 19999.2734375
Self consumption rate: 0.10228497714891982
Self sufficiency rate: 0.6727106297909017
Total cost: 218.94748647375565
Total revenue: 5571.342178638285
Net revenue: 5352.394692164529


In [110]:
print(SOC[-1])

0.0


In [111]:
print(fos)

0.0


In [94]:
dayo = datetime.strftime(day,"%Y-%m-%d") 

In [95]:
dayo

'2023-01-02'

In [124]:
linear_programming[linear_programming["soc"].isna()].shape[0]

1920

In [117]:
print(days[365])

2023-01-01
